In [0]:
%run /Workspace/Users/eyskanupuru@etihadppe.ae/azure_edp2_databricks/Production/eag/ey/AP_ANALYTICS/ZFIN_AP_CASHFLOW/schema_repository

In [0]:
%run /Workspace/Users/eyskanupuru@etihadppe.ae/azure_edp2_databricks/Production/eag/ey/AP_ANALYTICS/vendor_category_notebook

In [0]:
%run /Workspace/Users/eyskanupuru@etihadppe.ae/azure_edp2_databricks/Production/eag/ey/AP_ANALYTICS/Lead_and_Processor_notebook

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType
import pyspark.sql.functions as F
from pyspark.sql.functions import lit ,col, when, size, udf, collect_list, concat

today = datetime.now()
current_year = today.strftime("%Y")
last_year = str(int(current_year)-1)
last_to_last_year = str(int(current_year)-2)

In [0]:
lead_df.createOrReplaceTempView('lead')
processor_df.createOrReplaceTempView('processor')

In [0]:
base_source_path = "/mnt/stppeedp/ppeedp/landing/raw/finance/ap/AP_OUTSTATION_V2/FULL_INGESTIONS/"
unified_ingestion_path = "/mnt/stppeedp/ppeedp/landing/raw/finance/ap/UNIFIED_INGESTION/"
output_path = "/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/"
reference_path = "/mnt/stppeedp/ppeedp/landing/raw/finance/ap/REFERENCE_TABLES/"

In [0]:
#ACDOCA
table_name = 'ACDOCA'
csv_path = unified_ingestion_path + table_name + '/*.txt'
df=spark.read.option("header", "false").option("delimiter", "|||").schema(acdoca_schema_unified_ingestion).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('acdoca0')
spark.sql(f""" select distinct * from acdoca0 where (BUDAT like '{current_year}%' or BUDAT like '{last_year}%') and LIFNR is not null """ ).createOrReplaceTempView('acdoca')

In [0]:
spark.sql(""" select distinct GJAHR as sample_fiscal_year, RBUKRS as sample_company_code,  BELNR as sample_document_number, BLART as sample_document_type, concat(BELNR, RBUKRS, GJAHR, AUGBL, LIFNR, BUDAT) as concat_key, 'NA' as month from acdoca0 """).createOrReplaceTempView('sample')

In [0]:
#BSEG
table_name = 'BSEG'
csv_path = unified_ingestion_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(bseg_schema_ap_outstation_as_source).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('bsegV0')
spark.sql(f""" select * from bsegV0 where GJAHR in ('{current_year}', '{last_year}', '{last_to_last_year}') """).createOrReplaceTempView('bseg')

In [0]:
#BKPF.
table_name = 'BKPF'
csv_path = base_source_path + table_name + '/*.txt'
df=spark.read.option("header", "false").option("delimiter", "|||").schema(bkpf_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('bkpf')

In [0]:
#PRPS
table_name = 'PRPS'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(prps_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('prps')

In [0]:
#LFBK
table_name = 'LFBK'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(lfbk_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('lfbk')

In [0]:
#LFB1
table_name = 'LFB1'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(lfb1_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('lfb1')

In [0]:
#LFA1
table_name = 'LFA1'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(lfa1_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('lfa1')

In [0]:
#EKPO
table_name = 'EKPO'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(ekpo_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('ekpo')

In [0]:
#EKBE
table_name = 'EKBE'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(ekbe_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('ekbe')

In [0]:
#EKKN
table_name = 'EKKN'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(ekkn_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('ekkn')

In [0]:
#EKKO
table_name = 'EKKO'
csv_path = reference_path + table_name + '/*.txt'
df=spark.read.option("header", "false").schema(ekko_schema).csv(csv_path)
df = df.dropDuplicates()
df.createOrReplaceTempView('ekko')

In [0]:
prps1 = spark.sql(""" select distinct PSPNR from prps """)
prps1.createOrReplaceTempView('prps1') 

# ekbe1 = spark.sql(""" select distinct EBELN, EBELP, GJAHR, BELNR, DMBTR, REFWR, BUDAT from ekbe """)
# ekbe1.createOrReplaceTempView('ekbe1')

###################

ekbe0 = spark.sql(""" select distinct EBELN, EBELP, GJAHR from ekbe """) 
ekbe0.createOrReplaceTempView('ekbe0')

ekbe1 = spark.sql(""" select distinct EBELN, EBELP, BELNR from ekbe """)
ekbe1.createOrReplaceTempView('ekbe1')

ekbe2 = spark.sql(""" select distinct EBELN, EBELP, DMBTR from ekbe """)
ekbe2.createOrReplaceTempView('ekbe2')

ekbe3 = spark.sql(""" select distinct EBELN, EBELP, BUDAT from ekbe """)
ekbe3.createOrReplaceTempView('ekbe3')

ekbe4 = spark.sql(""" select distinct EBELN, EBELP, REFWR from ekbe """)
ekbe4.createOrReplaceTempView('ekbe4')

ekbe_x = spark.sql(""" select distinct EBELN as b_EBELN, EBELP as b_EBELP, REFWR as b_REFWR, BUDAT as b_BUDAT, DMBTR as b_DMBTR, BELNR as b_BELNR, GJAHR as b_GJAHR from ekbe """)
ekbe_x.createOrReplaceTempView('ekbe_x')

###################

ekko1 = spark.sql(""" select distinct AEDAT, ZTERM, WAERS, EBELN from ekko """)
ekko1.createOrReplaceTempView('ekko1')

ekpo1 = spark.sql(""" select distinct NETWR, EBELN, EBELP from ekpo """)
ekpo1.createOrReplaceTempView('ekpo1')

ekkn1 = spark.sql(""" select distinct EBELN, EBELP, SAKTO from ekkn """)
ekkn1.createOrReplaceTempView('ekkn1')

bkpf1 = spark.sql(""" select distinct XREF1_HD, GJAHR, BUKRS, BLART, BELNR, CPUDT, CPUTM, LOGSYSTEM_SENDER, BKTXT from bkpf """)
bkpf1.createOrReplaceTempView('bkpf1')

bseg1 = spark.sql(""" select distinct ZTERM, ZFBDT, H_BLART, GJAHR, BUKRS, DMBE3, BUZEI, NETDT, WMWST, BELNR, DMBTR, KOSTL, AUGCP, ZLSPR, ZLSCH, HBKID, VBUND from bseg """)
bseg1.createOrReplaceTempView('bseg1')

lfa12 = spark.sql(""" select distinct NAME1, LAND1, LIFNR from lfa1 """)
lfa12.createOrReplaceTempView('lfa12')

lfb12 = spark.sql(""" select distinct ZTERM, LIFNR from lfb1 """)
lfb12.createOrReplaceTempView('lfb12')

lfbk1 = spark.sql(""" select distinct BANKL, BANKN, LIFNR from lfbk """)
lfbk1.createOrReplaceTempView('lfbk1')

In [0]:
#Pass ACDOCA_AUBGL into ACDOCA_BELNR and ACDOCA_KOART = 'K' and ACDOCA_BSCHL = 50 and get GKONT
ac6 = spark.sql(""" select distinct BELNR,  BUZEI, KOART, BSCHL, GKONT FROM acdoca0 where KOART = 'K'  and BSCHL = '50' and BELNR is not null """ )
ac6 = ac6.dropDuplicates()
ac6.createOrReplaceTempView('acdoca6')

#Pass ACDOCA_AUBGL into ACDOCA_BELNR and ACDOCA_KOART = 'K' and ACDOCA_BSCHL = 50 and get RWCUR
ac7 = spark.sql(""" select distinct BELNR, BUZEI,  KOART, BSCHL, RWCUR FROM acdoca0 where KOART = 'K'  and BSCHL = '50' and BELNR is not null """ )
ac7 = ac7.dropDuplicates()
ac7.createOrReplaceTempView('acdoca7')

# Pass ACDOCA_AUBGL into ACDOCA_BELNR and ACDOCA_KOART = 'K' and ACDOCA_BSCHL = 50 and get HBKID
ac8= spark.sql(""" select distinct BELNR, BUZEI,  KOART, BSCHL, HBKID FROM acdoca0 where KOART = 'K'  and BSCHL = '50' and BELNR is not null """ )
ac8 = ac8.dropDuplicates()
ac8.createOrReplaceTempView('acdoca8')

# Pass ACDOCA_BELNR and ACDOCA_BSCHL = 31, get RACCT
ac1 = spark.sql(""" select distinct BELNR, BUZEI, BSCHL, RACCT FROM acdoca0 where BELNR is not null and BSCHL = '31' """ )
ac1 = ac1.dropDuplicates()
ac1.createOrReplaceTempView('acdoca1')

# Pass ACDOCA_BELNR and ACDOCA_KTOSL = WRX, get RACCT
ac2 = spark.sql(""" select distinct BELNR, BUZEI, KTOSL, RACCT  FROM acdoca0 where BELNR is not null and KTOSL = 'WRX' """ )
ac2 = ac2.dropDuplicates()
ac2.createOrReplaceTempView('acdoca2')

# Pass ACDOCA_EBELN and ACDOCA_KTOSL = WRX and get RACCT
ac3 = spark.sql(""" select distinct EBELN, BUZEI,  KTOSL, RACCT  FROM acdoca0 where EBELN is not null and KTOSL = 'WRX' """ )
ac3 = ac3.dropDuplicates()
ac3.createOrReplaceTempView('acdoca3')

# Pass ACDOCA_EBELN and ACDOCA_KTOSL = KBL and get RACCT
ac4 = spark.sql(""" select distinct EBELN, BUZEI, KTOSL, RACCT  FROM acdoca0 where EBELN is not null and KTOSL = 'KBS' """ )
ac4 = ac4.dropDuplicates()
ac4.createOrReplaceTempView('acdoca4')

In [0]:
spark.read.parquet('dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/vendor_category_20241127/').createOrReplaceTempView("vendor_category0")
spark.sql("""select distinct Vendor as Code, Category from vendor_category0 where Category is not null and Category <> '' """).createOrReplaceTempView("vendor_category")

In [0]:
# QUERY AP_OUTSTATION
query = f"""select
    lk.BANKL AS `Bank Account`,
    ac.BUZEI AS `Line item`, --ac.DOCLN AS `Line item`,
    ac.GJAHR AS `Fiscal Year`,
    ac.RBUKRS AS `Company Code`,
    ac.LIFNR AS `Vendor`,
    lf.NAME1 AS `Vendor Account: Name 1`,
    ac.BELNR AS `Document Number`,
    ac.BLDAT AS `Document Date`,
    ac.HSL AS `Company Code Currency Value`,
    ac.RHCUR AS `Company Code Currency Key`, --NA
    ac.AWREF AS `Reference`, 
    ac.BLART AS `Document type`,
    cast(bsg.NETDT as string) AS `Net Due date`, 
    ac.RWCUR AS `Document Currency Key`,
    ac.WSL AS `Document Currency Value`,
    ac.BSCHL AS `Posting Key`, --NA
    ac.AUGBL AS `Clearing Document`,
    bsg.ZFBDT AS `Baseline Payment Date`, --NA
    ac.AUGDT AS `Clearing Date`,
    bsg.AUGCP AS `Clearing Entry Date`,
    ac.BUDAT AS `Posting Date`,
    bkf.CPUDT AS `Entry Date`, --NA
    bkf.CPUTM AS `Time of Entry`, --NA
    'TBD' AS `Created by`, --NA
    ac.USNAM AS `User Name`, --NA
    bsg.ZLSPR AS `Payment Block`, --NA
    lb.ZTERM AS `Terms of Payment LFB1`, 
    bsg.ZTERM AS `Terms of Payment BSEG`,
    ekko.ZTERM AS `Terms of Payment EKKO`,
    bsg.ZLSCH AS `Payment Method`, --NA
    bsg.HBKID AS `House bank`, --NA
    ac.RACCT AS `GL\Account`,
    coalesce(lf.LAND1, ac.LAND1) as `Country`,
    'TBD' AS `City`,
    bkf.LOGSYSTEM_SENDER AS `Logical System of the Sender`, --NA
    ac.AWSYS AS `Logical System`,
    bkf.BKTXT AS `Document Header Text`,
    ac.SGTXT AS `Text`,
    bsg.VBUND AS `Trading partner`,
    ac.EBELN AS `Purchasing Document`,
    cat.Category AS `Categorization`,
    l.Lead AS `AP Lead`,
    p.Processor AS `AP Processor`,
    concat_ws('~', ac.GJAHR, ac.RBUKRS, ac.BELNR, ac.BUDAT, ac.BUZEI, ac.AUGBL, ac.AUGDT, ac.LIFNR, ac.WSL, ac.HSL, ac.OSL, ac.KSL) as key 
from 
    acdoca ac 
left join 
    bseg1 bsg on ac.BELNR = bsg.BELNR and ac.RBUKRS = bsg.BUKRS and ac.GJAHR = bsg.GJAHR and ac.BUZEI = bsg.BUZEI and ac.BLART = bsg.H_BLART 
left join 
    bkpf1 bkf on ac.BELNR = bkf.BELNR and ac.RBUKRS = bkf.BUKRS and ac.GJAHR = bkf.GJAHR and ac.BLART = bkf.BLART 
left join 
    ekko1 ekko on ac.EBELN = ekko.EBELN 
--------
left join 
    ekbe_x ekbx on ac.EBELN = ekbx.b_EBELN and ac.EBELP = ekbx.b_EBELP and ac.BUDAT = ekbx.b_BUDAT 
--------
left join
    ekpo1 ekpo on ac.EBELN = ekpo.EBELN and ac.EBELP = ekpo.EBELP ---line item(buzei = buzei)
left join
    ekkn1 ekkn on ac.EBELN = ekkn.EBELN and ac.EBELP = ekkn.EBELP 
left join 
    lfa12 lf on ac.LIFNR = lf.LIFNR
left join 
    lfb12 lb on ac.LIFNR = lb.LIFNR
left join 
    lfbk1 lk on ac.LIFNR = lk.LIFNR
left join 
    prps1 pr on ac.PS_POSID = pr.PSPNR
left join 
    acdoca3 ac3 on ac.EBELN = ac3.EBELN  and ac.BUZEI = ac3.BUZEI  ---line item id (buzei)
left join 
    acdoca4 ac4 on ac.EBELN = ac4.EBELN and ac.BUZEI = ac4.BUZEI  ---line item id (buzei)
left join 
    acdoca1 ac1 on ac.BELNR = ac1.BELNR and ac.BUZEI = ac1.BUZEI  ---line item id (buzei)
left join 
    acdoca2 ac2 on ac.BELNR = ac2.BELNR and ac.BUZEI = ac2.BUZEI  ---line item id (buzei)
left join 
    acdoca6 ac6 on ac.AUGBL = ac6.BELNR and ac.BUZEI = ac6.BUZEI  ---line item id (buzei)
left join 
    acdoca7 ac7 on ac.AUGBL = ac7.BELNR and ac.BUZEI = ac7.BUZEI  ---line item id (buzei)
left join 
    acdoca8 ac8 on ac.AUGBL = ac8.BELNR and ac.BUZEI = ac8.BUZEI  ---line item id (buzei)
left join 
    vendor_category cat on REGEXP_REPLACE(ac.LIFNR, '^0+', '') = REGEXP_REPLACE(cat.Code, '^0+', '')
left join 
    lead l on REGEXP_REPLACE(ac.LIFNR, '^0+', '') = REGEXP_REPLACE(l.Code, '^0+', '')
left join
    processor p on REGEXP_REPLACE(ac.LIFNR, '^0+', '') = REGEXP_REPLACE(p.Code, '^0+', '')
left join 
    sample sp on ac.BELNR = sp.sample_document_number and ac.RBUKRS = sp.sample_company_code and ac.GJAHR = sp.sample_fiscal_year and ac.BLART = sp.sample_document_type
"""

In [0]:
spark.sql(query).createOrReplaceTempView("APV1")
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY key ORDER BY `Posting Date`) AS row_number
        FROM APV1
    ) AS temp_table
    WHERE row_number = 1
""").createOrReplaceTempView("APV2")
query2 = """
SELECT distinct 
    `Bank Account`,
    `Line item`,
    `Fiscal Year`,
    `Company Code`,
    `Vendor`,
    `Vendor Account: Name 1`,
    `Document Number`,
    `Document Date`,
    `Company Code Currency Value`,
    `Company Code Currency Key`,
    `Reference`,
    `Document type`,
    `Net due date`,
    `Document Currency Key`,
    `Document Currency Value`,
    `Posting Key`,
    `Clearing Document`,
    `Baseline Payment Date`,
    `Clearing Date`,
    `Clearing Entry Date`,
    `Posting Date`,
    `Entry Date`,
    `Time of Entry`,
    `Created by`,
    `User Name`,
    `Payment Block`,
    `Terms of Payment LFB1`,
    `Terms of Payment BSEG`,
    `Terms of Payment EKKO`,
    `Payment Method`,
    `House bank`,
    `GL\Account`,
    `Country`,
    `City`,
    `Logical System of the Sender`,
    `Logical System`,
    substr(`Document Header Text`, 1, 4000) as `Document Header Text`,
    substr(`Text`, 1, 4000) as `Text`,
    `Trading partner`,
    `Purchasing Document`,
    `Categorization`,
    `AP Lead`,
    `AP Processor`,
    `key`
from APV2
"""

In [0]:
#STAGING TABLE WRITE
report_df = spark.sql(query2)
Staging_Path= output_path + "acdoca_ap_outstation/"
report_df = report_df.repartition(1)
report_df.write.mode("overwrite").parquet(Staging_Path)
spark.read.parquet(Staging_Path).createOrReplaceTempView("acdoca_source")

In [0]:
spark.sql("""select distinct `key` as ek, `Terms of Payment EKKO` as ez from acdoca_source where `Terms of Payment EKKO` is not null """ ).createOrReplaceTempView("ekz")
spark.sql("""select distinct `key` as lk, `Terms of Payment LFB1` as lz from acdoca_source where `Terms of Payment LFB1` is not null """ ).createOrReplaceTempView("lfz")
spark.sql("""select distinct `key` as bk, `Terms of Payment BSEG` as bz from acdoca_source where `Terms of Payment BSEG` is not null """ ).createOrReplaceTempView("bsz")
spark.sql("""select distinct `key` as pkey, `Purchasing Document` as pod from acdoca_source where `Purchasing Document` is not null """ ).createOrReplaceTempView("po")

query2 = """
SELECT distinct 
    `Bank Account`,
    `Line item`,
    `Fiscal Year`,
    `Company Code`,
    `Vendor`,
    `Vendor Account: Name 1`,
    `Document Number`,
    `Document Date`,
    `Company Code Currency Value`,
    `Company Code Currency Key`,
    `Reference`,
    `Document type`,
    `Net due date`,
    `Document Currency Key`,
    `Document Currency Value`,
    `Posting Key`,
    `Clearing Document`,
    `Baseline Payment Date`,
    `Clearing Date`,
    `Clearing Entry Date`,
    `Posting Date`,
    `Entry Date`,
    `Time of Entry`,
    `Created by`,
    `User Name`,
    `Payment Block`,
    coalesce(`Terms of Payment LFB1`, lz) as `Terms of Payment LFB1`,
    coalesce(`Terms of Payment BSEG`, bz) as `Terms of Payment BSEG`,
    coalesce(`Terms of Payment EKKO`, ez) as `Terms of Payment EKKO`,
    `Payment Method`,
    `House bank`,
    `GL\Account`,
    `Country`,
    `City`,
    `Logical System of the Sender`,
    `Logical System`,
    `Document Header Text`,
    `Text`,
    `Trading partner`,
    coalesce(`Purchasing Document`, pod) as `Purchasing Document`,
    `Categorization`,
    `AP Lead`,
    `AP Processor` 
from acdoca_source a left join ekz e on a.`key` = e.ek 
left join bsz b on a.`key` = b.bk
left join lfz l on a.`key` = l.lk
left join po p on a.`key` = p.pkey
"""

In [0]:
report_df = spark.sql(query2)
reprt_df = report_df.dropDuplicates()
Prod_Final_Path = output_path + "ap_outstation_v2/"
report_df = report_df.repartition(1)
report_df.write.mode("overwrite").parquet(Prod_Final_Path)